In [1]:
import langchain
print(langchain.__version__)

1.4.6


In [2]:
# 📦 Install if not already done


# 1️⃣ Imports
from langchain_core.tools import Tool
from langchain_classic.agents import initialize_agent, AgentType
from langchain_community.vectorstores import FAISS
#from langchain.embeddings import OpenAIEmbeddings 
from langchain_classic.chains import RetrievalQA
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_classic.memory import VectorStoreRetrieverMemory

import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
#from langchain.chains import LLMChain
from langchain_classic.memory import ConversationBufferMemory


# 2️⃣ Load API keys
load_dotenv(".env")
google_api_key=os.getenv("GOOGLE_API_KEY")

# 3️⃣ Setup LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite")

# 4️⃣ Create Vector DB (Retriever)

with open("sample.txt", "r", encoding="utf-8") as f:
    text_data = f.read()

# 🧠 Split the text into smaller chunks

from langchain_text_splitters import CharacterTextSplitter

splitter = CharacterTextSplitter(separator="\n", chunk_size=300, chunk_overlap=50)
texts = splitter.split_text(text_data)
embedding = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001")
vectorstore = FAISS.from_texts(texts, embedding)
retriever = vectorstore.as_retriever()

# 5️⃣ Create RetrievalQA Chain
qa_chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)

# 6️⃣ Convert to LangChain Tool
qa_tool = Tool(
    name="LangChainRetriever",
    func=qa_chain.run,
    description="Use this to answer questions about LangChain framework, features, or its creator."
)

# 7️⃣ Setup Memory (to support conversational context)
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# 8️⃣ Initialize Agent with Tool + Memory
agent = initialize_agent(
    tools=[qa_tool],
    llm=llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,  # ReAct agent
    memory=memory,
    verbose=True,
    handle_parsing_errors=True
)

# 9️⃣ Ask Questions (RAG-Style)
print("1️⃣ First Question")
res1 = agent.run("What is LangChain?")
print("Answer:", res1)

print("\n2️⃣ Follow-up")
res2 = agent.run("Who created it?")
print("Answer:", res2)

print("\n3️⃣ Combined Reasoning")
res3 = agent.run("Explain LangChain's use in AI workflows.")
print("Answer:", res3)


C:\Users\user\AppData\Local\Temp\ipykernel_11040\4000266392.py:56: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\pydantic\main.py:263: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  validated_self = self.__pydantic_validator__.validate_python(data, self

1️⃣ First Question


> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: LangChainRetriever
Action Input: What is LangChain?
Observation: LangChain is a framework for building applications with LLMs.
Thought:Do I need to use a tool? No
AI: LangChain is a framework designed to help developers build applications using Large Language Models (LLMs).

> Finished chain.
Answer: LangChain is a framework designed to help developers build applications using Large Language Models (LLMs).

2️⃣ Follow-up


> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: LangChainRetriever
Action Input: Who created LangChain?
Observation: LangChain was created by Harrison Chase.
Thought:Do I need to use a tool? No
AI: LangChain was created by Harrison Chase.

> Finished chain.
Answer: LangChain was created by Harrison Chase.

3️⃣ Combined Reasoning


> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: LangChainRetrie